# 04 · Baseline: simple RAG contract review

The **comparison point** for the adaptive agent. Per the project proposal, this is "a simple RAG system without an agentic layer" (§8) and it ships as its own notebook (§10).

**What it does.** For each contract and each of the six clause families:

1. Retrieve the top 4 chunks of the contract, using the family's **Keywords** line from `rubric_v1.md` as the query.
2. Send those chunks, the rubric's general rules (§0.2–0.4) and that family's rubric section to the model in **one call**.
3. Keep the verdict, severity, quote and one-sentence recommendation it returns. No confidence score, no follow-up, no second look, no tools.

**What it deliberately does not have** — these are the agentic layer, and the point of the comparison: rule routing beyond the fixed family → rubric lookup, a verification loop, aggregation into a prioritized plan, and bounded retries. The only retry here is `llm()` re-sending a request after an **API or network** error; the model's answer is never re-asked.

**Kept identical to the agent** (`02_agent_adk.ipynb`) so the comparison measures the agentic layer and not the plumbing: the same model (GPT-5 nano via Portkey), the same chunking (110 words, 30 overlap), the same retriever, the same rubric parser, the same prompt perspective, and the same `results.csv` columns.

**Two things the agent notebook still needs in order to match:**

1. the Retriever in §4 below (the stem fix). Without it the agent retrieves worse passages than this baseline — on the four dev contracts, 10 of 24 contract × family rows come back different.
2. the rubric's general rules (§0.2–0.4) in its `RAG_SYSTEM` prompt, which the pre-pass already has.

**Retrieval mode:** `RETRIEVAL_MODE = "hybrid"` (BM25 + embeddings), the same as the agent's pre-pass. If the course key can't make embeddings, both fall back to BM25; the mode actually used is saved in `summary.json`.

**Outputs** (in `runs/baseline_<split>/`): `results.csv` (one row per contract × family), `retrieved.jsonl` (the passages each call saw plus the raw reply), `summary.json` (calls, tokens, cost, time, retrieval mode).

Tune on the **dev contracts only** (C01, C02, C11, C12). `run_baseline` refuses test contracts unless `allow_test=True` (final scoring run only).

## 0. Setup

Colab secrets (🔑 on the left, notebook access on): `OPENAI_API_KEY` (the course Portkey key) and `GH_TOKEN` (to clone the private repo for `contracts/`, `contracts.csv` and `rubric_v1.md`).

In [ ]:
!pip -q install rank_bm25 openai

import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    if not Path("contract-review-agent").exists():
        token = userdata.get("GH_TOKEN")
        !git clone -q https://{token}@github.com/JJJJeny/contract-review-agent.git
    DATA_DIR = Path("contract-review-agent")
else:
    DATA_DIR = Path(".")          # run from the repo folder
print(sorted(p.name for p in DATA_DIR.iterdir()))

## 1. Model (Portkey → GPT-5 nano)

`llm(system, user)` returns `(text, input_tokens, output_tokens)`. It retries twice on API errors.

In [ ]:
import time
from openai import OpenAI

api_key = userdata.get("OPENAI_API_KEY") if IN_COLAB else os.environ.get("OPENAI_API_KEY")
MODEL = "@30875-fall26-openai/gpt-5-nano"
base_url = "https://api.portkey.ai/v1"
MODEL_PRICE_KEY = "gpt-5-nano"          # key into PRICES below
client = OpenAI(api_key=api_key, base_url=base_url) if api_key else None


def llm(system, user, tries=3):
    for attempt in range(tries):
        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
            u = r.usage
            return (r.choices[0].message.content or ""), (u.prompt_tokens if u else 0), (u.completion_tokens if u else 0)
        except Exception as e:
            if attempt == tries - 1:
                raise
            print(f"API error ({str(e)[:120]}); retrying")
            time.sleep(2 ** (attempt + 1))

print("Model:", MODEL, "| key loaded:", bool(api_key))

## 2. Settings

Same values as the agent notebook's `BOUNDS` where they overlap.

In [ ]:
FAMILIES = ["payment", "termination", "liability_indemnity",
            "confidentiality", "renewal", "dispute_governing_law"]

# USD per 1M tokens (input, output). Check current OpenAI pricing before reporting costs.
PRICES = {"gpt-5-nano": (0.05, 0.40), "gpt-4.1-nano": (0.10, 0.40), "fake": (0.0, 0.0)}

TOP_K = 4                  # chunks per family (agent: BOUNDS.prepass_top_k)
RETRIEVAL_MODE = "hybrid"  # "sparse" | "dense" | "hybrid"; agent pre-pass uses "hybrid"
CHUNK_WORDS, CHUNK_OVERLAP = 110, 30   # Contract.load defaults, same as the agent

## 3. Contract helpers (copied from the agent notebook)

Loads a contract, finds numbered headings, cuts it into chunks, and checks quotes (`contains`, `locate`).

In [ ]:
import re
from dataclasses import dataclass, field
from pathlib import Path

# A numbered heading such as "9." "9.2" "12.1.3" at the start of a line or sentence,
# followed by a capital letter or "(" (so "30 days" or "Section 9.2" are not headings).
HEADING = re.compile(r'(?:(?<=\n)|(?<=^)|(?<=\. )|(?<=: ))[ \t]*(\d{1,2}(?:\.\d{1,2}){0,3})\.?[ \t]+(?=[A-Z(“"])')
ATTACHMENT = re.compile(r'\b(EXHIBIT|SCHEDULE|APPENDIX|ANNEX|Exhibit|Schedule|Appendix|Annex)\s+["“]?([A-Z0-9]{1,2})\b["”]?')
CROSS_REF = re.compile(
    r'(?:\b(?:Section|Sections|Paragraph|Clause|Article)\s+\d{1,2}(?:\.\d{1,2})*(?:\s*\([a-z0-9]{1,4}\))*'
    r'|§\s*\d{1,2}(?:\.\d{1,2})*'
    r'|\b(?:Exhibit|Schedule|Appendix|Annex|EXHIBIT|SCHEDULE)\s+["“]?[A-Z0-9]{1,2}\b["”]?'
    r'|\bnotwithstanding\b|\bexcept as (?:set forth|provided|otherwise)|\bsubject to (?:Section|Article|Paragraph|Clause|Exhibit|Schedule)\b)',
    re.I)
DEFINED = re.compile(r'[("“]\s*(?:the\s+)?["“]([A-Z][A-Za-z0-9\-]*(?:\s+[A-Z][A-Za-z0-9\-]*){0,4})["”]')
MEANS = re.compile(r'\b([A-Z][A-Za-z\-]*(?:\s+[A-Z][A-Za-z\-]*){0,4})["”]?\s+(?:shall\s+)?(?:means?|has the meaning)\b')
ROLE_TERMS = {"Customer", "Provider", "Recipient", "Service", "Services", "Company", "Contractor", "Client", "Vendor"}
TRIVIAL_TERMS = {"Agreement", "Party", "Parties", "Effective Date", "Section", "Exhibit", "Schedule"}


def norm(s) -> str:
    s = s or ""
    return re.sub(r"\s+", " ", s.replace("“", '"').replace("”", '"').replace("’", "'")).strip()


@dataclass
class Chunk:
    idx: int
    start: int
    end: int
    section: str
    text: str


@dataclass
class Contract:
    cid: str
    text: str
    meta: dict = field(default_factory=dict)
    headings: list = field(default_factory=list)   # [(pos, "9.2")]
    chunks: list = field(default_factory=list)
    defined_terms: dict = field(default_factory=dict)  # term -> definition sentence
    preamble_terms: set = field(default_factory=set)

    @classmethod
    def load(cls, path, meta=None, chunk_words=110, overlap=30):
        text = Path(path).read_text(encoding="utf-8", errors="replace")
        c = cls(cid=Path(path).stem, text=text, meta=meta or {})
        c.headings = [(m.start(1), m.group(1)) for m in HEADING.finditer(text)]
        c._chunk(chunk_words, overlap)
        c._index_terms()
        return c

    def section_at(self, pos: int) -> str:
        sec = ""
        for p, s in self.headings:
            if p > pos:
                break
            sec = s
        return f"§{sec}" if sec else "untitled"

    def _chunk(self, size, overlap):
        words = [(m.start(), m.end()) for m in re.finditer(r"\S+", self.text)]
        step = size - overlap
        for i in range(0, max(1, len(words)), step):
            span = words[i:i + size]
            if not span:
                break
            s, e = span[0][0], span[-1][1]
            self.chunks.append(Chunk(len(self.chunks), s, e, self.section_at(s), norm(self.text[s:e])))
            if i + size >= len(words):
                break

    def _sentence_around(self, pos: int, max_chars=600) -> str:
        left = max(self.text.rfind(". ", 0, pos), self.text.rfind("\n\n", 0, pos))
        right = self.text.find(". ", pos)
        right = len(self.text) if right == -1 else right + 1
        return norm(self.text[left + 1:right])[:max_chars]

    def _index_terms(self):
        for rx in (DEFINED, MEANS):
            for m in rx.finditer(self.text):
                term = m.group(1).strip()
                if term in TRIVIAL_TERMS or len(term) < 3 or term in self.defined_terms:
                    continue
                self.defined_terms[term] = self._sentence_around(m.start())
                if m.start() < 1500:          # parties and roles are introduced in the preamble
                    self.preamble_terms.add(term)

    def contains(self, quote: str) -> bool:
        """verify_quote: does this exact text (whitespace-normalised) appear in the contract?"""
        q = norm(quote or "").strip(' ."')
        return bool(q) and q.lower() in norm(self.text).lower()

    def locate(self, quote: str):
        """Character span [start, end) of a quote in the original text (whitespace-insensitive), or None."""
        q = norm(quote).strip(' ."')
        if not q:
            return None
        if not hasattr(self, "_flat"):
            # normalised copy of the text plus a map from each normalised character to its original position
            flat, pos, prev_space = [], [], False
            for i, ch in enumerate(self.text.replace("“", '"').replace("”", '"').replace("’", "'")):
                if ch.isspace():
                    if not prev_space and flat:
                        flat.append(" ")
                        pos.append(i)
                    prev_space = True
                else:
                    flat.append(ch)
                    pos.append(i)
                    prev_space = False
            self._flat, self._pos = "".join(flat).lower(), pos
        k = self._flat.find(q.lower())
        if k < 0:
            return None
        return [self._pos[k], self._pos[k + len(q) - 1] + 1]

    def section_text(self, section_id: str, max_words=350) -> dict:
        sid = section_id.strip()
        att = ATTACHMENT.search(sid)
        if att or re.match(r'(?i)(exhibit|schedule|appendix|annex)', sid):
            return self._attachment_text(sid, max_words)
        num = re.search(r'\d{1,2}(?:\.\d{1,2})*', sid)
        if not num:
            return {"found": False, "note": f"could not parse section id {section_id!r}"}
        num = num.group(0)
        hits = [i for i, (p, s) in enumerate(self.headings) if s == num]
        if not hits:
            return {"found": False, "section": num,
                    "note": "no heading with this number; the reference may be wrong or the section may be in a missing exhibit"}
        i = hits[0]
        start = self.headings[i][0]
        depth = num.count(".")
        end = len(self.text)
        for p, s in self.headings[i + 1:]:
            if s.count(".") <= depth:
                end = p
                break
        words = norm(self.text[start:end]).split()
        return {"found": True, "section": f"§{num}", "text": " ".join(words[:max_words]),
                "truncated": len(words) > max_words}

    def _attachment_text(self, sid, max_words):
        m = re.search(r'(?i)(exhibit|schedule|appendix|annex)\s+["“]?([A-Z0-9]{1,2})', sid)
        if not m:
            return {"found": False, "note": f"could not parse attachment id {sid!r}"}
        kind, label = m.group(1).lower(), m.group(2).upper()
        rx = re.compile(rf'(?i)\b{kind}\s+["“]?{re.escape(label)}\b["”]?')
        mentions, body_starts = [], []
        for h in rx.finditer(self.text):
            before = self.text[max(0, h.start() - 40):h.start()]
            ctx = norm(self.text[max(0, h.start() - 80):h.end() + 80])
            mentions.append(ctx)
            # Treat it as the attachment itself when written as a heading (upper case, or on its own line).
            after = self.text[h.end():h.end() + 3]
            is_heading = self.text[h.start():h.end()].isupper() or before.rstrip(" \t").endswith("\n")
            # Attachments are filed after the body, so ignore table-of-contents lines ("Schedule A: Fees").
            past_body = not self.headings or h.start() > self.headings[-1][0]
            if (is_heading and past_body and not after.lstrip().startswith(":")
                    and not re.search(r'(?i)(in|to|on|of|per|under|the|see)\s*$', before)):
                body_starts.append(h.start())
        if body_starts:
            start = body_starts[-1]
            words = norm(self.text[start:]).split()
            return {"found": True, "section": f"{kind.title()} {label}", "text": " ".join(words[:max_words]),
                    "truncated": len(words) > max_words}
        return {"found": False, "section": f"{kind.title()} {label}",
                "note": "EXHIBIT NOT INCLUDED: only references to it appear in the filed text",
                "mentions": mentions[:3]}

    def define(self, term: str) -> dict:
        t = term.strip(' "“”')
        for k, v in self.defined_terms.items():
            if k.lower() == t.lower():
                return {"found": True, "term": k, "definition": v}
        # fallback: first sentence where the term appears in quotes or followed by "means"
        m = re.search(rf'["“]{re.escape(t)}["”]|{re.escape(t)}\s+(?:shall\s+)?means', self.text, re.I)
        if m:
            return {"found": True, "term": t, "definition": self._sentence_around(m.start())}
        return {"found": False, "term": t, "note": "no definition found; the term may be defined in a missing exhibit"}


def cross_refs(text: str) -> list:
    return sorted({norm(m.group(0)) for m in CROSS_REF.finditer(text)})


def undefined_terms(text: str, contract: Contract, already_resolved=()) -> list:
    """Defined terms used in a span whose definition is not yet part of the evidence."""
    out = []
    for term in contract.defined_terms:
        if term in already_resolved:
            continue
        if term in contract.preamble_terms or term in ROLE_TERMS:
            continue
        if re.search(rf'\b{re.escape(term)}\b', text):
            out.append(term)
    return out

## 4. Retriever (agent's version + stem fix)

Identical to the agent notebook except `_expand`: a query word that never appears in the contract (e.g. the stem `terminat`) is replaced by the contract's words that start with it (`terminate`, `terminated`, `termination`). **Copy this cell into `02_agent_adk.ipynb` §4 as well.**

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

TOKEN = re.compile(r"[a-z0-9]+")


def tok(s):
    return TOKEN.findall(s.lower())


class Retriever:
    def __init__(self, contract, embedder=None):
        self.c = contract
        docs = [tok(ch.text) for ch in contract.chunks]
        self.bm25 = BM25Okapi(docs)
        self.vocab = {t for d in docs for t in d}
        self.embedder = embedder          # callable(list[str]) -> np.ndarray, or None
        self._vecs = None

    def _expand(self, query):
        """Stem fix: BM25 matches whole words, so a stem like "terminat" matched nothing.
        Replace each query word that is not in the contract by the contract words that start with it."""
        out = []
        for t in tok(query):
            if t in self.vocab or len(t) < 4:
                out.append(t)
            else:
                out += sorted(v for v in self.vocab if v.startswith(t)) or [t]
        return out

    def _dense_scores(self, query):
        if self._vecs is None:
            self._vecs = self.embedder([ch.text for ch in self.c.chunks])
        q = self.embedder([query])[0]
        return self._vecs @ q / (np.linalg.norm(self._vecs, axis=1) * np.linalg.norm(q) + 1e-9)

    def search(self, query, mode="hybrid", k=3, exclude=()):
        """Return the top-k chunks as dicts. mode: sparse | dense | hybrid."""
        if mode in ("dense", "hybrid") and self.embedder is None:
            mode = "sparse"               # no embeddings available: degrade, and say so in the result
        sparse = self.bm25.get_scores(self._expand(query))
        if mode == "sparse":
            score = sparse
        else:
            dense = self._dense_scores(query)
            if mode == "dense":
                score = dense
            else:
                rank = lambda a: np.argsort(np.argsort(-a))
                score = 1 / (60 + rank(sparse)) + 1 / (60 + rank(dense))
        order = [i for i in np.argsort(-score) if i not in exclude][:k]
        return mode, [{"chunk": int(i), "section": self.c.chunks[i].section,
                       "text": self.c.chunks[i].text} for i in order]

## 5. Embeddings for hybrid retrieval (copied from the agent notebook)

If the course key does not allow embeddings, this prints `dense retrieval off` and retrieval falls back to BM25.

In [ ]:
from openai import OpenAI

EMBED_MODEL = "@30875-fall26-openai/text-embedding-3-small"


def make_embedder():
    client = OpenAI(api_key=api_key, base_url=base_url)

    def embed(texts):
        out = []
        for i in range(0, len(texts), 100):
            r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 100])
            out += [d.embedding for d in r.data]
        return np.array(out, dtype=np.float32)

    try:
        embed(["test"])
        return embed
    except Exception as e:            # the course key may not allow embeddings: search falls back to sparse
        print("dense retrieval off:", str(e)[:200])
        return None


EMBEDDER = make_embedder() if client else None
print("dense retrieval:", "on" if EMBEDDER else "off")

## 6. Rubric (copied from the agent notebook)

Parses `rubric_v1.md`: each family's section, its **Keywords** line, and each sub-criterion row.

In [ ]:
import re
from pathlib import Path


def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")


class Rubric:
    def __init__(self, path):
        self.text = Path(path).read_text(encoding="utf-8")
        self.rules = {}           # rule_id -> text
        self.family_keywords = {}
        self._parse()

    def _parse(self):
        parts = re.split(r"^(#{2,3} .+)$", self.text, flags=re.M)
        for head, body in zip(parts[1::2], parts[2::2]):
            m = re.match(r"#{2,3} (\d+(?:\.\d+)?)\.? (.+)", head)
            if not m:
                continue
            num, title = m.group(1), m.group(2).strip()
            self.rules[num] = f"{head}\n{body.strip()}"
            fam = title.split()[0]
            if re.fullmatch(r"[a-z_]+", fam):              # "## 1. payment" etc.
                self.rules[fam] = self.rules[num]
                kw = re.search(r"\*\*Keywords:\*\* (.+)", body)
                if kw:
                    self.family_keywords[fam] = kw.group(1).rstrip(".")
                for row in re.findall(r"^\| ([^|]+?) \| (.+)$", body, flags=re.M):
                    name = row[0].strip()
                    if name.startswith("Sub-criterion") or set(name) <= set("-"):
                        continue
                    self.rules[f"{fam}.{slug(name)}"] = f"| {name} | {row[1]}"
                for label in ("Missing", "Severity", "Absence rule", "Customer is not the payer"):
                    p = re.search(rf"^\*\*{label}[^*]*\*\*.*$", body, flags=re.M)
                    if p:
                        self.rules[f"{fam}.{slug(label)}"] = p.group(0)

    def family_rules(self, family):
        return self.rules.get(family, "")

    def general_rules(self):
        return "\n\n".join(self.rules[k] for k in ("0.2", "0.3", "0.4") if k in self.rules)

    def lookup(self, rule_id):
        rid = rule_id.strip().lstrip("§").lower()
        if rid in self.rules:
            return {"found": True, "rule_id": rid, "text": self.rules[rid]}
        near = [k for k in self.rules if rid.split(".")[0] in k][:12]
        return {"found": False, "rule_id": rid, "available": near or sorted(self.rules)[:20]}


RUBRIC = Rubric(DATA_DIR / "data" / "rubric_v1.md")
print(len(RUBRIC.rules), "rules")

## 7. Token accounting and JSON parsing (copied from the agent notebook)

In [ ]:
import json
import re
import time

import numpy as np



class Usage:
    def __init__(self, model):
        self.model = model
        self.input_tokens = self.output_tokens = self.calls = 0

    def add(self, i, o):
        self.input_tokens += i
        self.output_tokens += o
        self.calls += 1
        return self.cost_of(i, o)

    def cost_of(self, i, o):
        pin, pout = PRICES.get(self.model, (0.0, 0.0))
        return (i * pin + o * pout) / 1e6

    @property
    def tokens(self):
        return self.input_tokens + self.output_tokens

    @property
    def cost(self):
        return self.cost_of(self.input_tokens, self.output_tokens)


def parse_json(text):
    """Parse a model reply that should be JSON. Never raises.

    Small models sometimes wrap the JSON in prose or send several objects back to back
    ("Extra data" errors). Every object in the text is decoded; "families" / "items" are
    merged across objects, otherwise the largest object wins.
    """
    text = text or ""
    dec, objs, i = json.JSONDecoder(), [], 0
    while True:
        i = text.find("{", i)
        if i < 0:
            break
        try:
            obj, i = dec.raw_decode(text, i)
            if isinstance(obj, dict):
                objs.append(obj)
        except json.JSONDecodeError:
            i += 1
    if not objs:
        print("warning: no JSON in model reply:", text[:200].replace("\n", " "))
        return {}
    for key in ("families", "items"):
        if any(key in o for o in objs):
            merged = {} if key == "families" else []
            for o in objs:
                part = o.get(key)
                if isinstance(merged, dict) and isinstance(part, dict):
                    merged.update(part)
                elif isinstance(merged, list) and isinstance(part, list):
                    merged += part
            return {key: merged}
    return max(objs, key=lambda o: len(json.dumps(o, default=str)))

## 8. Prompt

`RAG_SYSTEM` is the agent notebook's prompt plus two additions: the rubric's **general rules** (§0.2–0.4 — what each verdict means, the severity table, and the rule that `unclear` caps at medium) and a `recommendation` field.

Both additions are about fairness, not capability. Without §0.2–0.4 the baseline would be picking verdicts whose definitions it was never given while the agent's pre-pass has them, and the proposal scores "usefulness of recommendations" (§8), which needs the baseline to produce one at all.

**Add the same `{general}` block to `RAG_SYSTEM` in `02_agent_adk.ipynb` §7.**

In [ ]:
PERSPECTIVE = (
    "You review vendor service agreements for a SMALL-BUSINESS CUSTOMER that buys maintenance or "
    "services. It has no lawyer and little bargaining power. Judge every clause from the customer's "
    "side, even when the provider filed the contract. Apply the rubric exactly; do not invent thresholds."
)

RAG_SYSTEM = PERSPECTIVE + """

Baseline: read the retrieved passages for ONE clause family and judge it with the rubric below.

Rules that apply to every family:
{general}

Rules for this family:
{rules}

Return JSON only:
{{"verdict": "acceptable|risky|missing|unclear",
  "severity": "high|medium|low|n/a",
  "evidence_text": "exact sentence copied from a passage (empty if the clause is missing)",
  "section_ref": "the section as printed, e.g. 9.2; copy the section tag of the passage you quoted",
  "evidence_2": "second exact quote the verdict depends on, or empty",
  "section_ref_2": "",
  "recommendation": "one plain sentence the customer can say to the provider; empty if acceptable",
  "notes": "which sub-criterion decided it"}}"""

print(RAG_SYSTEM.format(general=RUBRIC.general_rules(), rules=RUBRIC.family_rules("renewal"))[:1800])

## 9. The baseline

`review_family` = retrieve → one call → parse. `review_contract` runs the six families. After the call, **code** (not the model):

- enforces rubric §0.4 on severity (`clean_severity`): `acceptable` → `n/a`, a flagged row always carries a severity, and `unclear` is capped at **medium**. The baseline cannot know whether "both readings are high", so it applies the cap and records in `notes` that it did;
- fills `section_ref` from where the quote sits in the contract when the model left it blank (`section_for`);
- for a `missing` verdict, writes the keywords searched and the sections seen into `notes`. Rubric §0.5 requires this: a "missing" with no search record is treated as "not checked" during adjudication;
- checks each quote word for word against the contract (`evidence_verified`);
- records the character span of each quote (`evidence_spans`, for Track A overlap with CUAD).

The quote check only **records** whether a quote is real. Acting on a failed check is the agent's job, not the baseline's.

In [ ]:
VERDICTS = {"acceptable", "risky", "missing", "unclear"}
SEVERITIES = {"high", "medium", "low", "n/a"}


def clean_severity(verdict, severity):
    """Rubric §0.4, enforced in code. Returns (severity, note) so any change is auditable."""
    s = (severity or "").strip().lower()
    if verdict == "acceptable":
        return "n/a", ""
    if s not in SEVERITIES or s == "n/a":
        return "low", f" [severity {severity!r} not usable for a {verdict} row; set to low]"
    if verdict == "unclear" and s == "high":
        return "medium", " [§0.4: unclear capped at medium]"
    return s, ""


def section_for(contract, quote, given):
    """Keep the model's section ref if it gave one, otherwise read it off where the quote sits."""
    if (given or "").strip():
        return given.strip()
    sp = contract.locate(quote or "")
    return contract.section_at(sp[0]) if sp else ""


def review_family(contract, retriever, fam, llm, usage):
    query = RUBRIC.family_keywords.get(fam, fam)
    mode_used, hits = retriever.search(query, RETRIEVAL_MODE, TOP_K)
    user = json.dumps({"family": fam, "passages": hits}, ensure_ascii=False)
    text, i, o = llm(RAG_SYSTEM.format(general=RUBRIC.general_rules(), rules=RUBRIC.family_rules(fam)), user)
    usage.add(i, o)
    d = parse_json(text)
    notes = (d.get("notes") or "").strip()

    if d.get("verdict") not in VERDICTS:
        notes += f" [unparseable verdict {d.get('verdict')!r}; set to unclear]"
        d["verdict"] = "unclear"
    d["severity"], sev_note = clean_severity(d["verdict"], d.get("severity"))
    notes += sev_note

    for q_key, s_key in (("evidence_text", "section_ref"), ("evidence_2", "section_ref_2")):
        d[s_key] = section_for(contract, d.get(q_key), d.get(s_key))
        q = d.get(q_key) or ""
        d[q_key + "_verified"] = q in ("", "NONE FOUND") or contract.contains(q)

    if d["verdict"] == "missing":          # rubric §0.5: a missing verdict must show its search
        notes += f" [searched: {query}; sections seen: {', '.join(h['section'] for h in hits)}]"

    d["notes"] = notes.strip()
    d["recommendation"] = (d.get("recommendation") or "").strip()
    d["evidence_spans"] = [sp for sp in (contract.locate(d.get("evidence_text") or ""),
                                         contract.locate(d.get("evidence_2") or "")) if sp]
    return d, {"contract_id": contract.cid, "family": fam, "query": query,
               "mode_used": mode_used, "passages": hits, "raw_reply": text[:2000]}


def review_contract(contract, llm, embedder=None):
    retriever = Retriever(contract, embedder)
    usage = Usage(getattr(llm, "price_key", MODEL_PRICE_KEY))
    t0 = time.time()
    results, retrieved = {}, []
    for fam in FAMILIES:
        results[fam], r = review_family(contract, retriever, fam, llm, usage)
        retrieved.append(r)
    summary = {"contract_id": contract.cid, "llm_calls": usage.calls, "tokens": usage.tokens,
               "cost_usd": round(usage.cost, 5), "elapsed_s": round(time.time() - t0, 2),
               "retrieval_mode_used": sorted({r["mode_used"] for r in retrieved})}
    return results, retrieved, summary

## 10. Runner

Writes the agent's 21 `results.csv` columns in the same order, plus `recommendation` as a 22nd — the baseline's counterpart to the agent's `ask` / `fallback`, which live in its `revision_plans.json`. Scoring merges on `contract_id` + `family`, so the extra trailing column changes nothing.

Agent-only columns are filled with their "no loop" values: `path = rag`, `steps = 0`, `verdict_changed = False`, and blanks for confidence, pre-pass and triggers.

In [ ]:
import csv

COLUMNS = ["contract_id", "short_name", "split", "family", "verdict", "severity", "evidence_text",
           "section_ref", "evidence_2", "section_ref_2", "notes",
           "confidence", "path", "steps", "evidence_verified",
           "prepass_verdict", "prepass_severity", "prepass_confidence", "verdict_changed", "triggers_fired",
           "evidence_spans", "recommendation"]


def load_meta():
    with open(DATA_DIR / "data" / "contracts.csv", newline="") as f:
        return {r["contract_id"]: r for r in csv.DictReader(f)}


def run_baseline(ids, llm=llm, out_dir="runs/baseline_dev", embedder=None, allow_test=False):
    meta = load_meta()
    test_ids = [i for i in ids if meta[i]["split"] == "test"]
    if test_ids and not allow_test:
        raise ValueError(f"{test_ids} are test contracts. Tune on dev only; pass allow_test=True for the final scoring run.")
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows, retrieved, summary = [], [], []
    for cid in ids:
        contract = Contract.load(DATA_DIR / "data" / "contracts" / f"{cid}.txt", meta[cid], CHUNK_WORDS, CHUNK_OVERLAP)
        res, ret, s = review_contract(contract, llm, embedder)
        for fam in FAMILIES:
            f = res[fam]
            rows.append({"contract_id": cid, "short_name": meta[cid]["short_name"], "split": meta[cid]["split"],
                         "family": fam, **{k: f.get(k, "") for k in COLUMNS[4:11]},
                         "confidence": "", "path": "rag", "steps": 0,
                         "evidence_verified": f["evidence_text_verified"] and f["evidence_2_verified"],
                         "prepass_verdict": "", "prepass_severity": "", "prepass_confidence": "",
                         "verdict_changed": False, "triggers_fired": "",
                         "evidence_spans": json.dumps(f["evidence_spans"]),
                         "recommendation": f.get("recommendation", "")})
        retrieved += ret
        summary.append(s)
        print(s)
    with open(out_dir / "results.csv", "w", newline="") as f:
        w = csv.DictWriter(f, COLUMNS)
        w.writeheader()
        w.writerows(rows)
    with open(out_dir / "retrieved.jsonl", "w") as f:
        for r in retrieved:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    (out_dir / "summary.json").write_text(json.dumps(
        {"model": getattr(llm, "price_key", MODEL), "condition": "rag", "retrieval_mode": RETRIEVAL_MODE,
         "top_k": TOP_K, "chunk_words": CHUNK_WORDS, "chunk_overlap": CHUNK_OVERLAP,
         "contracts": summary}, indent=2))
    print("wrote", out_dir)
    return out_dir

## 11. Free check with a fake model

No API calls. Checks retrieval, the stem fix, parsing, the §0.4 severity rules, the `section_ref` backfill, the search record on a `missing` verdict, the output columns and the dev-only guard. Run it after any change.

In [ ]:
import re


def fake_llm(system, user):
    """Quotes the first sentence of the first passage and calls it risky. Costs nothing.
    Leaves section_ref blank on purpose, so the backfill is exercised."""
    ps = json.loads(user)["passages"]
    q = re.split(r"(?<=[.;])\s", ps[0]["text"])[0][:300] if ps else ""
    return json.dumps({"verdict": "risky", "severity": "medium", "evidence_text": q, "section_ref": "",
                       "recommendation": "Ask for a fairer term.", "notes": "fake"}), 1000, 100
fake_llm.price_key = "fake"


def fake_missing_llm(system, user):
    """Always answers "missing", to check the rubric §0.5 search record."""
    return json.dumps({"verdict": "missing", "severity": "high", "evidence_text": "",
                       "notes": "nothing found"}), 500, 50
fake_missing_llm.price_key = "fake"

DEV = ["C01", "C02", "C11", "C12"]

# the stem fix: "terminat" must now reach termination clauses
c01 = Contract.load(DATA_DIR / "data" / "contracts" / "C01.txt")
r01 = Retriever(c01)
assert "terminat" not in r01.vocab and any(t.startswith("terminat") for t in r01._expand("terminat"))
assert any(t.startswith("indemnif") for t in r01._expand("indemnif"))

# rubric §0.4, enforced in code
assert clean_severity("acceptable", "high") == ("n/a", "")
assert clean_severity("unclear", "high")[0] == "medium"
assert clean_severity("risky", "n/a")[0] == "low"
assert clean_severity("risky", "nonsense")[0] == "low"
assert clean_severity("risky", "HIGH ") == ("high", "")

d = run_baseline(DEV, fake_llm, "runs/check_baseline")
rows = list(csv.DictReader(open(d / "results.csv")))
assert len(rows) == len(DEV) * 6 and list(rows[0]) == COLUMNS
assert all(r["path"] == "rag" for r in rows)
assert sum(r["evidence_verified"] == "True" for r in rows) >= len(rows) * 0.8, "fake quotes come from passages, so most must verify"
assert all(r["section_ref"] for r in rows), "section_ref must be filled by the model or backfilled from the quote"
assert all(r["recommendation"] for r in rows), "a flagged row needs a recommendation"

# a "missing" verdict must record what was searched (rubric §0.5)
d2 = run_baseline(["C01"], fake_missing_llm, "runs/check_missing")
rows2 = list(csv.DictReader(open(d2 / "results.csv")))
assert all("searched:" in r["notes"] for r in rows2)
assert all(r["severity"] == "high" for r in rows2), "missing keeps its severity; only unclear is capped"

try:
    run_baseline(["C03"], fake_llm, "runs/check_guard")
    raise AssertionError("test contract was not refused")
except ValueError:
    pass
print("all checks passed")

## 12. Real run on the dev contracts

Start with one contract to check the connection and cost, then all four dev contracts.

In [ ]:
out_dir = run_baseline(["C01"], llm, "runs/baseline_C01", embedder=EMBEDDER)

In [ ]:
out_dir = run_baseline(DEV, llm, "runs/baseline_dev", embedder=EMBEDDER)

## 13. Quick look

In [ ]:
import pandas as pd

res = pd.read_csv(out_dir / "results.csv")
summ = pd.DataFrame(json.load(open(out_dir / "summary.json"))["contracts"])
print(f"calls {summ.llm_calls.sum()}  tokens {summ.tokens.sum()}  cost ${summ.cost_usd.sum():.4f}  "
      f"time {summ.elapsed_s.sum():.0f}s  retrieval {sorted(set(sum(summ.retrieval_mode_used, [])))}")
print(f"quotes verified {res.evidence_verified.mean():.0%}  |  "
      f"section refs filled {(res.section_ref.fillna('') != '').mean():.0%}  |  "
      f"rows with a recommendation {(res.recommendation.fillna('') != '').mean():.0%}  |  "
      f"unparseable replies {int(res.notes.fillna('').str.contains('unparseable').sum())}")
display(pd.crosstab(res.family, res.verdict, margins=True))
res[["contract_id", "family", "verdict", "severity", "section_ref", "evidence_verified", "recommendation"]]

## 14. Scoring against the two answer keys (dev only here)

Two keys, because they answer different questions:

- **Track B, risk verdicts** → `labels_gold.xlsx` (sheet `labels`): the frozen file both annotators adjudicated. Gives agreement, **false alarms** (baseline flags risky/missing, gold says acceptable) and **missed risks** (baseline says acceptable, gold says risky/missing), broken down by family. These are the headline numbers in proposal §8.
- **Track A, clause detection** → `cuad_gold_spans.csv`: does a baseline quote overlap a CUAD-highlighted span?

CUAD covers only **38 of the 90** contract × family cells and has no category at all for payment or confidentiality, so it cannot score risk verdicts. That is why Track B needs our own labels, and it is the change from the proposal's original plan to use CUAD as the single ground truth. The cell prints the covered-cell count so the report can state it.

Run this on dev while tuning. The final numbers come from the test contracts, with `allow_test=True`.

In [ ]:
GOLD = DATA_DIR / "data" / "labels_gold.xlsx"        # frozen and adjudicated: the Track B answer key
SPANS = DATA_DIR / "data" / "cuad_gold_spans.csv"    # CUAD highlights: the Track A answer key
FLAG = {"risky", "missing"}

if GOLD.exists():
    gold = pd.read_excel(GOLD, sheet_name="labels", keep_default_na=False)[
        ["contract_id", "family", "verdict", "severity"]]
    cmp = res.merge(gold, on=["contract_id", "family"], suffixes=("_rag", "_gold"))
    cmp["agree"] = cmp.verdict_rag == cmp.verdict_gold
    cmp["false_alarm"] = cmp.verdict_rag.isin(FLAG) & (cmp.verdict_gold == "acceptable")
    cmp["missed_risk"] = (cmp.verdict_rag == "acceptable") & cmp.verdict_gold.isin(FLAG)
    print(f"Track B on {len(cmp)} rows  |  agreement {cmp.agree.mean():.0%}  |  "
          f"false alarms {int(cmp.false_alarm.sum())}  |  missed risks {int(cmp.missed_risk.sum())}")
    display(cmp.groupby("family")[["agree", "false_alarm", "missed_risk"]].sum())
    display(pd.crosstab(cmp.verdict_gold, cmp.verdict_rag))
else:
    print("no gold labels at", GOLD)

if SPANS.exists():
    cuad = pd.read_csv(SPANS)
    cuad["end"] = cuad.answer_start + cuad.text.str.len()
    print(f"\nCUAD covers {cuad.groupby(['contract_id', 'family']).ngroups} of 90 contract x family cells; "
          f"no CUAD category for {', '.join(sorted(set(FAMILIES) - set(cuad.family)))}")
    rows_a = []
    for r in res.itertuples():
        g = cuad[(cuad.contract_id == r.contract_id) & (cuad.family == r.family)]
        if g.empty:
            continue                                  # CUAD has no answer here: not scored in Track A
        spans = json.loads(r.evidence_spans)
        hit = any(s < ge and gs < e for s, e in spans for gs, ge in zip(g.answer_start, g.end))
        rows_a.append({"contract_id": r.contract_id, "family": r.family, "found_cuad_clause": hit})
    track_a = pd.DataFrame(rows_a)
    print(f"Track A, quote overlaps a CUAD span: {track_a.found_cuad_clause.mean():.0%} "
          f"of {len(track_a)} scored rows")
    display(track_a.groupby("family").found_cuad_clause.mean())
else:
    print("no CUAD spans at", SPANS)

## 15. Download the run

In [ ]:
if IN_COLAB:
    !zip -qr baseline_run.zip {out_dir}
    from google.colab import files
    files.download("baseline_run.zip")